<a href="https://colab.research.google.com/github/AMBOT-pixel96/hr-tech-portfolio/blob/main/people_analytics/HR_DataForge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install faker pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 24.0 MB/s eta 0:00:00


In [3]:
from faker import Faker
import pandas as pd
import numpy as np
import random

fake = Faker()
Faker.seed(42)
np.random.seed(42)

departments = ["Finance", "HR", "IT", "Marketing", "Operations"]
job_levels = ["Analyst", "Associate", "Manager", "Senior Manager", "Director"]
genders = ["Male", "Female"]

records = []
for i in range(200):
    emp_id = f"E{1000+i}"
    dept = random.choice(departments)
    level = random.choice(job_levels)
    gender = random.choice(genders)
    perf_rating = np.clip(np.random.normal(3.2, 0.9), 1, 5).round(1)
    ctc = np.random.normal(7 if level=="Manager" else 4, 1.5) * 1e5
    records.append([emp_id, dept, level, gender, round(perf_rating,1), abs(ctc)])

df_perf = pd.DataFrame(records, columns=["EmployeeID","Department","JobLevel","Gender","PerformanceRating","CTC"])
df_perf.to_csv("Performance_Data.csv", index=False)
df_perf.head()

,EmployeeID,Department,JobLevel,Gender,PerformanceRating,CTC
0,E1000,Operations,Director,Female,3.6,379260.354824
1,E1001,IT,Associate,Female,3.8,628454.478461
2,E1002,Finance,Director,Male,3.0,364879.456458
3,E1003,IT,Director,Female,4.6,515115.209373
4,E1004,IT,Director,Female,2.8,481384.006538


In [4]:
departments = ["Finance", "HR", "IT", "Marketing", "Operations"]
job_levels = ["Analyst", "Associate", "Manager", "Senior Manager"]
genders = ["Male", "Female"]

records = []
for i in range(150):
    emp_id = f"E{2000+i}"
    dept = random.choice(departments)
    level = random.choice(job_levels)
    gender = random.choice(genders)
    # More senior = slightly higher scores
    base = 3.5 + 0.2 * job_levels.index(level)
    q = [np.clip(np.random.normal(base, 0.7), 1, 5) for _ in range(5)]
    records.append([emp_id, dept, level, gender, *[round(x,1) for x in q]])

df_eng = pd.DataFrame(records, columns=["EmployeeID","Department","JobLevel","Gender","Q1","Q2","Q3","Q4","Q5"])
df_eng.to_csv("Engagement_Survey.csv", index=False)
df_eng.head()

,EmployeeID,Department,JobLevel,Gender,Q1,Q2,Q3,Q4,Q5
0,E2000,Finance,Associate,Female,2.6,3.3,3.7,3.7,3.4
1,E2001,Finance,Associate,Female,4.1,3.0,3.6,3.8,4.1
2,E2002,Finance,Analyst,Female,4.0,2.7,2.4,4.4,3.7
3,E2003,HR,Analyst,Female,3.0,4.6,3.6,4.3,3.5
4,E2004,Operations,Analyst,Female,4.9,4.7,3.3,4.2,4.0


In [5]:
job_roles = ["Analyst","Manager","Senior Manager","Director"]
bench_records = []
emp_records = []

for i in range(150):
    emp_id = f"E{3000+i}"
    dept = random.choice(departments)
    role = random.choice(job_roles)
    level = role
    gender = random.choice(genders)
    perf = np.clip(np.random.normal(3, 1), 1, 5)
    base_ctc = np.random.normal(6 + 1.2*job_roles.index(role), 1.2) * 1e5
    bonus = base_ctc * np.random.uniform(0.05, 0.25)
    emp_records.append([emp_id, gender, dept, role, level, round(base_ctc,0), round(bonus,0), round(perf,1)])

for role in job_roles:
    bench_records.append([role, role, round(np.random.normal(7 + 1.3*job_roles.index(role), 0.8)*1e5, 0)])

df_comp = pd.DataFrame(emp_records, columns=["EmployeeID","Gender","Department","JobRole","JobLevel","CTC","Bonus","PerformanceRating"])
df_bench = pd.DataFrame(bench_records, columns=["JobRole","JobLevel","MarketMedianCTC"])

df_comp.to_csv("Compensation_Internal.csv", index=False)
df_bench.to_csv("Compensation_Benchmark.csv", index=False)
df_comp.head()

,EmployeeID,Gender,Department,JobRole,JobLevel,CTC,Bonus,PerformanceRating
0,E3000,Male,Finance,Director,Director,701593.0,68412.0,2.0
1,E3001,Male,Finance,Manager,Manager,647883.0,41879.0,1.7
2,E3002,Female,Finance,Analyst,Analyst,721178.0,125411.0,4.6
3,E3003,Female,Finance,Director,Director,878041.0,195511.0,1.0
4,E3004,Female,IT,Director,Director,1234913.0,68079.0,1.0


In [6]:

# ============================================
# 🩸 Attrition Dataset — Realistic Version (v2.0)
# ============================================

import pandas as pd, numpy as np, random
from faker import Faker
from google.colab import files

fake = Faker()
np.random.seed(42)
Faker.seed(42)

departments = ["Finance", "HR", "IT", "Marketing", "Operations"]
job_levels = ["Analyst", "Associate", "Manager", "Senior Manager", "Director"]
genders = ["Male", "Female"]

exit_reasons = [
    "Better Pay", "Relocation", "Career Change",
    "Health Reasons", "Personal", "Manager Conflict",
    "Retirement", "Layoff"
]

records = []
for i in range(250):
    emp_id = f"E{4000 + i}"
    dept = random.choice(departments)
    level = random.choice(job_levels)
    gender = random.choice(genders)

    # --- Tenure (realistic skew: most between 1–5 yrs, some 8–10)
    tenure_months = int(np.clip(np.random.normal(36 + 6*job_levels.index(level), 18), 3, 120))

    # --- CTC linked to level
    ctc = np.random.normal(5 + 1.3*job_levels.index(level), 1.5) * 1e5

    # --- Attrition logic: newer employees → higher attrition
    attrition_prob = np.clip(0.35 - (tenure_months/240), 0.05, 0.35)
    attrition = np.random.choice(["Yes", "No"], p=[attrition_prob, 1 - attrition_prob])

    # --- Exit Reason (only for attrited employees)
    exit_reason = random.choice(exit_reasons) if attrition == "Yes" else ""

    records.append([
        emp_id, dept, level, gender, tenure_months, abs(round(ctc, 0)), attrition, exit_reason
    ])

df_attr = pd.DataFrame(records, columns=[
    "EmployeeID", "Department", "JobLevel", "Gender",
    "TenureMonths", "CTC", "AttritionFlag", "ExitReason"
])

# Save & auto-download
df_attr.to_csv("Attrition_Data.csv", index=False)
files.download("Attrition_Data.csv")
print("✅ Attrition_Data.csv generated & downloaded successfully!")
df_attr.head(10)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Attrition_Data.csv generated & downloaded successfully!


,EmployeeID,Department,JobLevel,Gender,TenureMonths,CTC,AttritionFlag,ExitReason
0,E4000,Finance,Director,Male,68,999260.0,No,
1,E4001,Finance,Director,Male,39,1067835.0,Yes,Manager Conflict
2,E4002,Operations,Associate,Male,70,745115.0,Yes,Manager Conflict
3,E4003,IT,Director,Female,49,941225.0,No,
4,E4004,Finance,Senior Manager,Male,58,603008.0,No,
5,E4005,Operations,Analyst,Female,50,271419.0,Yes,Layoff
6,E4006,Operations,Manager,Male,31,548154.0,No,
7,E4007,Finance,Analyst,Female,24,589658.0,No,
8,E4008,HR,Analyst,Male,26,516638.0,No,
9,E4009,Operations,Director,Female,49,1162116.0,Yes,Retirement


In [7]:

# ============================================
# 🏢 Workforce Dataset — Realistic Version (v2.0)
# ============================================

import pandas as pd, numpy as np, random
from faker import Faker
from google.colab import files

fake = Faker()
np.random.seed(42)
Faker.seed(42)

departments = ["Finance", "HR", "IT", "Marketing", "Operations"]
job_levels = ["Analyst", "Associate", "Manager", "Senior Manager", "Director"]
genders = ["Male", "Female"]

# Department-wise plausible job roles
roles_by_dept = {
    "Finance": ["Analyst", "Accounts Executive", "Finance Manager", "Auditor"],
    "HR": ["Recruiter", "HRBP", "L&D Specialist", "HR Manager"],
    "IT": ["Developer", "System Engineer", "Tech Lead", "Data Analyst"],
    "Marketing": ["Brand Executive", "Digital Marketer", "SEO Specialist", "Marketing Manager"],
    "Operations": ["Operations Analyst", "Team Lead", "Process Manager", "Coordinator"]
}

skills_map = {
    "Finance": ["Excel", "PowerBI", "Accounting", "Risk Analysis"],
    "HR": ["HRIS", "Recruitment", "Onboarding", "Analytics"],
    "IT": ["Python", "SQL", "Machine Learning", "Cloud"],
    "Marketing": ["SEO", "Content", "Campaigns", "CRM"],
    "Operations": ["Lean", "Six Sigma", "Supply Chain", "Automation"]
}

records = []
for i in range(200):
    emp_id = f"E{5000 + i}"
    dept = random.choice(departments)
    level = random.choice(job_levels)
    gender = random.choice(genders)
    manager_id = f"M{random.randint(1, 25):03}"

    # --- Role (mapped to department)
    job_role = random.choice(roles_by_dept[dept])

    # --- TenureMonths with mild level correlation
    base_tenure = np.random.normal(24 + 8*job_levels.index(level), 10)
    tenure_months = int(np.clip(base_tenure, 3, 120))

    # --- CTC (scaled to level)
    ctc = np.random.normal(5 + 1.2*job_levels.index(level), 1.3) * 1e5

    # --- Skillset (random 2–3 skills)
    skillset = "; ".join(random.sample(skills_map[dept], k=2))

    records.append([
        emp_id, dept, job_role, level, gender, manager_id, tenure_months, abs(round(ctc, 0)), skillset
    ])

df_work = pd.DataFrame(records, columns=[
    "EmployeeID", "Department", "JobRole", "JobLevel",
    "Gender", "ManagerID", "TenureMonths", "CTC", "Skills"
])

# Save & auto-download
df_work.to_csv("Workforce_Data.csv", index=False)
files.download("Workforce_Data.csv")
print("✅ Workforce_Data.csv generated & downloaded successfully!")
df_work.head(10)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Workforce_Data.csv generated & downloaded successfully!


,EmployeeID,Department,JobRole,JobLevel,Gender,ManagerID,TenureMonths,CTC,Skills
0,E5000,Operations,Team Lead,Director,Male,M020,60,962026.0,Lean; Automation
1,E5001,IT,System Engineer,Director,Male,M002,62,1177994.0,Python; SQL
2,E5002,Finance,Analyst,Manager,Female,M005,37,709562.0,PowerBI; Excel
3,E5003,Marketing,Marketing Manager,Director,Male,M012,71,1079767.0,Campaigns; Content
4,E5004,HR,Recruiter,Senior Manager,Male,M021,43,930533.0,Recruitment; Onboarding
5,E5005,IT,System Engineer,Director,Female,M021,51,919455.0,SQL; Python
6,E5006,HR,Recruiter,Director,Female,M001,58,731274.0,HRIS; Analytics
7,E5007,Marketing,SEO Specialist,Analyst,Male,M004,6,426903.0,Campaigns; CRM
8,E5008,Operations,Process Manager,Associate,Female,M012,21,660852.0,Automation; Supply Chain
9,E5009,Marketing,Marketing Manager,Senior Manager,Female,M007,38,676401.0,Content; SEO


In [ ]:
from google.colab import files

# List all generated CSVs
csv_files = [
    "Performance_Data.csv",
    "Engagement_Survey.csv",
    "Compensation_Internal.csv",
    "Compensation_Benchmark.csv",
    "Attrition_Data.csv",
    "Workforce_Data.csv"
]

# Download one by one
for f in csv_files:
    try:
        files.download(f)
        print(f"✅ Downloaded {f}")
    except Exception as e:
        print(f"⚠️ Could not download {f}: {e}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloaded Performance_Data.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

✅ Downloaded Attrition_Data.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloaded Workforce_Data.csv
